# OCR BNP — Pipeline V12 — Qwen3.6-27B-FP8 + finegrained-fp8 V1 local
> Version adaptée à Domino **sans modification du dépôt finegrained-fp8 V1**.

**Compatibilité ajoutée :**
- Qwen3.6-27B-FP8 local Domino
- `kernels==0.16.x` / `transformers 5.15.x`
- finegrained-fp8 **V1** local via adaptateur Python
- DeepGEMM désactivé (fallback Triton, car `nvcc` complet n'est pas disponible)
- aucun téléchargement réseau du kernel FP8
- logique métier V12, JSON checkpoints, batch GPU, statistiques et Excel conservés

> Exécuter avec **Kernel → Restart Kernel and Run All Cells** afin que les variables d'environnement soient définies avant l'import de Transformers.


## 1. Vérification de l'environnement (aucune installation automatique)


In [ ]:
import sys
import importlib.metadata as md

print("Python      :", sys.version.split()[0])
for pkg in ["torch", "torchvision", "transformers", "kernels", "triton", "accelerate", "safetensors", "tokenizers"]:
    try:
        print(f"{pkg:12s}: {md.version(pkg)}")
    except Exception:
        print(f"{pkg:12s}: ABSENT")

# Versions attendues dans l'environnement Domino actuellement utilisé.
# On ne fait aucun pip install ici pour ne pas modifier l'environnement.
try:
    kver = md.version("kernels")
    major_minor = tuple(int(x) for x in kver.split(".")[:2])
    if major_minor != (0, 16):
        raise RuntimeError(f"kernels {kver} détecté ; cette version du notebook attend kernels 0.16.x")
except Exception as e:
    print("ATTENTION kernels :", e)

## 2. Imports + adaptateur finegrained-fp8 V1


In [ ]:
import os, sys, time, json, re
import numpy as np
from pathlib import Path
from datetime import datetime
from collections import defaultdict
import fitz
import torch
from PIL import Image

# ------------------------------------------------------------------
# Kernel FP8 LOCAL Domino (v1)
# ------------------------------------------------------------------
LOCAL_FP8 = "/domino/edv/modelhub/ModelHub-model-huggingface-kernels-community/finegrained-fp8/main/build/torch-cuda"

if not Path(LOCAL_FP8).exists():
    raise FileNotFoundError(f"Kernel FP8 local introuvable : {LOCAL_FP8}")
if LOCAL_FP8 not in sys.path:
    sys.path.insert(0, LOCAL_FP8)

import finegrained_fp8
import kernels

# ------------------------------------------------------------------
# PATCH LARGE : intercepte TOUT appel de chargement pour finegrained-fp8
# Corrige le "ValueError" car transformers 5.3.0 ne passe pas de version.
# ------------------------------------------------------------------
def _make_wrapper(orig_fn):
    def wrapper(*args, **kwargs):
        # Récupère le repo_id (1er argument positionnel ou kwarg)
        repo_id = args[0] if args else kwargs.get("repo_id", "")
        if isinstance(repo_id, str) and "finegrained-fp8" in repo_id:
            return finegrained_fp8
        return orig_fn(*args, **kwargs)
    return wrapper

# On patche toutes les fonctions publiques du module kernels (et kernels.hub si présent)
for _mod in [kernels, getattr(kernels, "hub", None)]:
    if _mod is None: continue
    for _attr_name in dir(_mod):
        if _attr_name.startswith("_"): continue
        _attr = getattr(_mod, _attr_name, None)
        if callable(_attr):
            setattr(_mod, _attr_name, _make_wrapper(_attr))

# ------------------------------------------------------------------
# IMPORTS TRANSFORMERS (après le patch)
# ------------------------------------------------------------------
from transformers import AutoProcessor, AutoModelForMultimodalLM
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

print("✅ Imports OK")
print("✅ Kernel FP8 local :", finegrained_fp8.__file__)
print("   fp8_act_quant     :", finegrained_fp8.fp8_act_quant)
print("   w8a8_block_matmul :", finegrained_fp8.w8a8_block_fp8_matmul)

In [ ]:
torch.manual_seed(0)
M_, K_, N_, bn, bk = 32, 512, 1024, 128, 128
x  = torch.randn(M_, K_, dtype=torch.bfloat16, device="cuda")
w  = (torch.randn(N_, K_, device="cuda") * 0.02).to(torch.bfloat16)
ws = w.float().abs().reshape(N_//bn, bn, K_//bk, bk).amax(dim=(1, 3))
wq = (w.float() / ws.repeat_interleave(bn, 0).repeat_interleave(bk, 1)).to(torch.float8_e4m3fn)
ref = x.float() @ (wq.float() * ws.repeat_interleave(bn, 0).repeat_interleave(bk, 1)).T
out = finegrained_fp8.matmul_2d(x, wq, ws, [bn, bk], torch.float32)
rel = ((out - ref).abs().max() / ref.abs().max()).item()
print(f"err rel vs fp32 : {rel:.4f}")   # attendu < ~0.02
assert rel < 0.05

## 3. Config


In [ ]:
MODEL_PATH      = "/domino/edv/modelhub/ModelHub-model-huggingface-Qwen/Qwen3.6-27B-FP8/main"
DEVICE          = "cuda" if torch.cuda.is_available() else "cpu"
MAX_NEW_TOKENS  = 700
IMAGE_MAX_SIZE  = 1120
PDF_ZOOM        = 2.0
BLANK_THRESHOLD = 0.97
GPU_BATCH_SIZE  = 2      # H100 80GB : commencer à 2, augmenter seulement après validation

INPUT_DIR  = Path("/mnt/data/transferts_in")
OUTPUT_DIR = Path("/mnt/data/transferts_out")
JSON_DIR   = OUTPUT_DIR / "json_dossiers"
LOG_PATH   = OUTPUT_DIR / "pipeline.log"
EXCEL_PATH = OUTPUT_DIR / f"audit_transferts_{datetime.now().strftime('%Y%m%d_%H%M')}.xlsx"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
JSON_DIR.mkdir(parents=True, exist_ok=True)

pdfs = sorted(INPUT_DIR.glob("*.pdf"))
print(f'Device         : {DEVICE}')
print(f'GPU batch size : {GPU_BATCH_SIZE}')
print(f'Dossiers       : {len(pdfs)}')
print(f'JSON dir       : {JSON_DIR}')
print(f'Excel          : {EXCEL_PATH}')

## 4. Chargement Qwen3.6-27B-FP8


In [ ]:
t0 = time.time()

processor = AutoProcessor.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True,
    local_files_only=True,
)

# Ne pas forcer torch_dtype=float16 : le checkpoint contient sa configuration FP8.
model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_PATH,
    device_map="auto",
    trust_remote_code=True,
    local_files_only=True,
    low_cpu_mem_usage=True,
)
model.eval()

# Avec un seul H100 visible et device_map="auto", le modèle est normalement sur cuda:0.
MODEL_DEVICE = model.device

print(f'✅ Modèle chargé en {time.time()-t0:.1f}s')
print(f'   Classe : {model.__class__.__name__}')
print(f'   Device : {MODEL_DEVICE}')
if torch.cuda.is_available():
    print(f'   GPU    : {torch.cuda.get_device_name(0)}')
    print(f'   VRAM allouée : {torch.cuda.memory_allocated(0)/1024**3:.2f} GB')

## 5. Utilitaires PDF & inférence Qwen3.6-VL


In [ ]:
def resize(img, max_side=IMAGE_MAX_SIZE):
    w, h = img.size
    if max(w, h) <= max_side:
        return img
    r = max_side / max(w, h)
    return img.resize((int(w*r), int(h*r)), Image.LANCZOS)


def is_blank(image, threshold=BLANK_THRESHOLD) -> bool:
    arr = np.array(image.convert('L'))
    return (arr > 240).sum() / arr.size >= threshold


def pdf_to_pages(path: Path, zoom=PDF_ZOOM) -> list:
    doc = fitz.open(path)
    matrix = fitz.Matrix(zoom, zoom)
    pages = []
    for i in range(len(doc)):
        pix = doc.load_page(i).get_pixmap(matrix=matrix, alpha=False)
        img = resize(Image.frombytes('RGB', [pix.width, pix.height], pix.samples))
        pages.append({'index': i, 'image': img})
    doc.close()
    return pages


def parse_json(text: str) -> dict:
    try:
        m = re.search(r'\{.*\}', text, re.S)
        return json.loads(m.group()) if m else {}
    except Exception:
        return {}


def _move_inputs_to_model(inputs):
    """Déplace seulement les tenseurs vers le device principal du modèle."""
    if hasattr(inputs, 'to'):
        return inputs.to(MODEL_DEVICE)
    return {
        k: (v.to(MODEL_DEVICE) if torch.is_tensor(v) else v)
        for k, v in inputs.items()
    }


# ── Inférence single (1 image) ────────────────────────────────────────────────
def ask_single(prompt: str, image: Image.Image) -> dict:
    """Traite 1 image. Retourne {text, tokens_in, tokens_out, elapsed}."""
    messages = [{
        'role': 'user',
        'content': [
            {'type': 'image', 'image': image},
            {'type': 'text', 'text': prompt},
        ],
    }]

    # Qwen3.6-VL : le processor construit directement texte + tenseurs vision.
    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors='pt',
    )
    inputs = _move_inputs_to_model(inputs)

    input_len = inputs['input_ids'].shape[1]
    t0 = time.time()
    with torch.inference_mode():
        out = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            use_cache=True,
        )
    elapsed = time.time() - t0

    generated = out[0][input_len:]
    text = processor.decode(
        generated,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True,
    )
    attn = inputs.get('attention_mask')
    tok_in = int(attn[0].sum().item()) if attn is not None else int(input_len)

    return {
        'text': text,
        'tokens_in': tok_in,
        'tokens_out': int(len(generated)),
        'elapsed': round(elapsed, 2),
    }


# ── Inférence batch (N images en parallèle) ───────────────────────────────────
def ask_batch(prompt: str, images: list) -> list:
    """Traite N images en un appel GPU et retourne une réponse par image."""
    if not images:
        return []
    if len(images) == 1:
        return [ask_single(prompt, images[0])]

    conversations = []
    for img in images:
        conversations.append([{
            'role': 'user',
            'content': [
                {'type': 'image', 'image': img},
                {'type': 'text', 'text': prompt},
            ],
        }])

    inputs = processor.apply_chat_template(
        conversations,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors='pt',
        padding=True,
    )
    inputs = _move_inputs_to_model(inputs)

    padded_input_len = inputs['input_ids'].shape[1]
    attn = inputs.get('attention_mask')

    t0 = time.time()
    with torch.inference_mode():
        out = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            use_cache=True,
        )
    elapsed_total = time.time() - t0

    results = []
    for i in range(len(images)):
        generated = out[i][padded_input_len:]
        text = processor.decode(
            generated,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True,
        )
        tok_in = int(attn[i].sum().item()) if attn is not None else int(padded_input_len)
        results.append({
            'text': text,
            'tokens_in': tok_in,
            'tokens_out': int(len(generated)),
            'elapsed': round(elapsed_total / len(images), 2),
        })
    return results


print('✅ Utilitaires Qwen3.6-VL OK')

### 5.1 Test rapide de l'adaptateur FP8 V1 (sans lancer le lot)


In [ ]:
# Ce test ne lance pas encore le modèle. Il vérifie que Transformers utilise bien
# le bundle V1 injecté et ne tente plus de charger les symboles V4 depuis le Hub.
_bundle = hf_fp8.load_finegrained_fp8_kernel()
assert _bundle is hf_fp8._FINEGRAINED_FP8
assert callable(_bundle.matmul)
assert callable(_bundle.batched_matmul)
assert callable(_bundle.grouped_matmul)
print("✅ Adaptateur FP8 V1 actif dans Transformers")
print("   matmul         :", _bundle.matmul)
print("   batched_matmul :", _bundle.batched_matmul)
print("   grouped_matmul :", _bundle.grouped_matmul)

## 6. Normalisation des données

In [ ]:
def norm_str(v) -> str:
    if v is None: return None
    s = re.sub(r'\s+', ' ', str(v).strip())
    return s if s and s.lower() not in ('null', 'none', 'n/a') else None

def norm_upper(v) -> str:
    s = norm_str(v)
    return s.upper() if s else None

def norm_compte(v) -> str:
    s = norm_str(v)
    if not s: return None
    return re.sub(r'[^A-Za-z0-9]', '', s).upper()

def norm_montant(v) -> float:
    if v is None: return None
    if isinstance(v, (int, float)): return float(v)
    s = str(v).strip()
    s = re.sub(r'[^\d.,]', '', s)
    if not s: return None
    if s.count(',') == 1 and '.' not in s: s = s.replace(',', '.')
    elif '.' in s and ',' in s:            s = s.replace(',', '')
    elif s.count(',') > 1:                 s = s.replace(',', '')
    try: return float(s)
    except: return None

def norm_date(v) -> str:
    if not v: return None
    s = norm_str(v)
    if not s: return None
    if re.match(r'^\d{2}/\d{2}/\d{4}$', s): return s
    m = re.match(r'^(\d{4})-(\d{2})-(\d{2})$', s)
    if m: return f'{m.group(3)}/{m.group(2)}/{m.group(1)}'
    return s

def norm_periode(v) -> str:
    if not v: return None
    s = str(v).upper().strip()
    s = re.sub(r'[._\-]', ' ', s)
    s = re.sub(r'PART\s*(\d)', r'PART \1', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s


# Whitelists — champs autorisés par type (anti-parasite)
CHAMPS_OV = {
    'type', 'monnaie', 'montant_chiffres', 'montant_lettres',
    'periode','mois','annee', 'tranche','complement', 'date_demande', 'compte_donneur_ordre','nature_paiement_autre_libelle',
    'beneficiaire_nom', 'beneficiaire_compte', 'beneficiaire_adresse',
    'code_swift_banque_beneficiaire', 'nom_banque_beneficiaire', 'complement_ov'
}
CHAMPS_ANN1 = {
    'type', 'nom_prenom_employe', 'date_naissance','résidence','compte_bancaire_local',
    'nom_prenom_signataire', 'date_signature'
}
CHAMPS_ANN2 = {
    'type', 'mois_transfert', 'nom_prenom_travailleur', 'compte_bancaire_local',
    'salaire_mensuel', 'nombre_jours', 'part_transferable',
    'pays_destination', 'nom_banque_etranger', 'numero_compte_devise_etranger',
    'numero_domiciliation'
}
CHAMPS_BP = {
    'type', 'nom_prenom_salarie', 'matricule', 'mois_bulletin',
    'salaire_base', 'salaire_brut', 'retenue_ss', 'retenue_irg',
    'retenue_mutuelle', 'net_a_payer'
}


def normalise_doc(doc_type: str, data: dict) -> dict:
    d = dict(data)

    if doc_type == 'OV':
        # Aplatissement zones imbriquées
        for key in list(d.keys()):
            if isinstance(d.get(key), dict):
                d.update(d.pop(key))
        # Fallback nom alternatif compte_debiteur
        if not d.get('compte_donneur_ordre'):
            for alias in ['compte_donneur_ordre', 'compte_donneur','compte_debiteur','compte_bancaire','Siége Racine Ordinal clé','N°DOM',
                          'numero_compte', 'compte_debiteur_ordre']:
                if d.get(alias):
                    d['compte_donneur_ordre'] = d[alias]
                    break
        # Corriger tranche = numéro de zone
        if d.get('tranche') in ('70','32','50','59','57', 70, 32, 50, 59, 57):
            d['tranche'] = None
        # Corriger periode = date (format dd/mm/yyyy)
        if d.get('periode') and re.match(r'^\d{2}/\d{2}/\d{4}$', str(d.get('periode',''))):
            d['periode'] = None
        # Whitelist
        d = {k: v for k, v in d.items() if k in CHAMPS_OV}
        # Normalisation
        d['monnaie']              = norm_upper(d.get('monnaie'))
        d['montant_chiffres']     = norm_montant(d.get('montant_chiffres'))
        d['montant_lettres']      = norm_str(d.get('montant_lettres'))
        d['periode']              = norm_periode(d.get('periode'))

        d['annee']              = norm_str(d.get('annee'))
        d['mois']                 = norm_periode(d.get('mois'))
        d['tranche']              = norm_str(d.get('tranche'))
        d['complement_ov']              = norm_str(d.get('complement_ov'))
        d['date_demande']         = norm_date(d.get('date_demande'))


        d['compte_donneur_ordre'] = norm_compte(d.get('compte_donneur_ordre'))
        d['nature_paiement_autre_libelle']              = norm_str(d.get('nature_paiement_autre_libelle'))

        d['beneficiaire_nom']     = norm_upper(d.get('beneficiaire_nom'))
        d['beneficiaire_compte']  = norm_compte(d.get('beneficiaire_compte'))
        d['beneficiaire_adresse'] = norm_str(d.get('beneficiaire_adresse'))
        d['code_swift_banque_beneficiaire'] = norm_compte(d.get('code_swift_banque_beneficiaire'))
        d['nom_banque_beneficiaire']     = norm_upper(d.get('nom_banque_beneficiaire'))


    elif doc_type == 'ANNEXE_I':
        d = {k: v for k, v in d.items() if k in CHAMPS_ANN1}
        d['nom_prenom_employe']    = norm_upper(d.get('nom_prenom_employe'))
        d['compte_bancaire_local']      = norm_compte(d.get('compte_bancaire_local'))
        d['date_naissance']       = norm_date(d.get('date_naissance'))
        d['résidence']              = norm_str(d.get('résidence'))
        d['nom_prenom_signataire']= norm_upper(d.get('nom_prenom_signataire'))
        d['date_signature']       = norm_date(d.get('date_signature'))

         



    

    elif doc_type == 'ANNEXE_II':
        d = {k: v for k, v in d.items() if k in CHAMPS_ANN2}
        d['Periode_transfert']         = norm_periode(d.get('Periode_transfert'))
        d['tranche_transfert']              = norm_str(d.get('tranche_transfert'))
        d['complement_transfert']              = norm_str(d.get('complement_transfert'))
        
        d['nom_prenom_travailleur'] = norm_upper(d.get('nom_prenom_travailleur'))
        d['compte_bancaire_local']        = norm_compte(d.get('compte_bancaire_local'))
        d['salaire_mensuel']        = norm_montant(d.get('salaire_mensuel'))
        d['nombre_jours']           = norm_str(d.get('nombre_jours'))
        d['nombre_jours_absence']           = norm_str(d.get('nombre_jours_absence'))

        
        d['part_transferable']      = norm_montant(d.get('part_transferable'))
        d['pays_destination']       = norm_upper(d.get('pays_destination'))
        d['nom_banque_etranger']             = norm_str(d.get('nom_banque_etranger'))
        d['numero_compte_devise_etranger'] = norm_compte(d.get('numero_compte_devise_etranger'))
        d['numero_domiciliation']   = norm_str(d.get('numero_domiciliation'))




    
    elif doc_type == 'BULLETIN':
        d = {k: v for k, v in d.items() if k in CHAMPS_BP}
        d['nom_prenom_salarie'] = norm_upper(d.get('nom_prenom_salarie'))
        d['matricule']          = norm_str(d.get('matricule'))
        d['mois_bulletin']      = norm_periode(d.get('mois_bulletin'))
        d['salaire_base']       = norm_montant(d.get('salaire_base'))
        d['salaire_brut']       = norm_montant(d.get('salaire_brut'))
        d['retenue_ss']         = norm_montant(d.get('retenue_ss'))
        d['retenue_irg']        = norm_montant(d.get('retenue_irg'))
        d['retenue_mutuelle']   = norm_montant(d.get('retenue_mutuelle'))
        d['net_a_payer']        = norm_montant(d.get('net_a_payer'))

    return d


print('✅ Normalisation OK')

## 7. Prompt universel

In [ ]:
PROMPT_UNIVERSEL = """\
Lis ce document bancaire.

ÉTAPE 1 — Identifie le type en lisant le titre principal :
- OV        : titre "ORDRE DE VIREMENT A L'ETRANGER"

- ANNEXE_II : titre "Annexe II", contient "Fiche de paie spéciale relative à un transfert du salaire"
- BULLETIN  : titre "BULLETIN DE PAIE"
- ANNEXE_I  : le titre est exactement "Annexe I", (JAMAIS "ANNEXE II")
C'est une lettre adressée au Directeur de l'agence BNP.
Elle commence par "Je soussigné..." et se termine par deux signatures (L'Empployeur et le Requérant).
Elle Ne contient Pas de  tableau de salaire, pas de montants de retenues.

- AUTRE : tout autre docuement (page vide , tableau , email....)

ÉTAPE 2 — Selon le type identifié, extrais uniquement les champs listés ci-dessous.
Si le type est AUTRE, retourne uniquement {"type": "AUTRE"}.

--- Si OV ---
Zone 32 : monnaie, montant_chiffres, montant_lettres
Zone 70 : periode (mois + année  jamais ecrit sous JJ/MM/AAAA : elle est ecrite  JANVIER 2026 / null si absent ), mois (JANV , JAN, JANVIER / null si absent) ,annee (2026 , 26 / null si absent), tranche (apres la periode : P1 , PART1 PARTIE 1 / null si absent ), complement_ov(apres la periode : COM , COMP , COMPLEMENT / null si absent )
          tranche (Part 1 / Part 2 / null. Ne pas confondre avec le numéro 70 imprimé à gauche)
Zone 50 : date_demande, compte_donneur_ordre ("Siege Racine Ordinal clé" ou "N°DOM").
Zone nature_paiement :  nature_paiement_case (unr seul valeur parmi : virement_comemercial,virement_trésorerie, urgent, non urgent), nature_paiement_autre_libelle (texte EXACT entre parenthese a coté de la case Autre, ex :"(Trasnfert en USD)" ou "(Trasnfert en EURO)", sinon Null) 
Zone 59 : beneficiaire_nom, beneficiaire_compte (IBAN sans espaces), beneficiaire_adresse
Zone 57 : code_swift_banque_beneficiaire (code bic), nnom_banque_beneficiaire.





--- Si ANNEXE_II ---

mois_transfert (valeur brute après "Mois de"),
nom_prenom_travailleur
compte_bancaire_local (numéro 20 chiffres),
salaire_mensuel,
nombre_jours (apres le texte EXACT "Nombre de jour" / null si absent),
nombre_jours_absence (apres le texte EXACT "Nombre de jour d'absence" / null si absent),
part_transferable ,
pays_destination,
nom_banque_etranger(première partie après "Compte devise :"),
numero_compte_devise_etranger(IBAN après nom_banque_etranger),
numero_domiciliation : lire toute la ligne du tableau réctangulaire BNP situé a la page sous "DOMICILIATION IMPORT" . le tableau contient 5 colonnes sur une seule ligne. Retourner les 5 valeur séparées par | exactement comme elles apparaiessent. Exemple : 271901|YYYY.N|NN|NNNNN|DZD / null si tableau absent,vide ou masqué par un cachet).





--- Si ANNEXE_I ---
Prends le temps de lire toute la lettre avant d'extraire.
nom_prenom_employe : le nom juste aprés "Je soussigné" en debut de lettre,
date_naissance : Né le jj/MM/aaaa
résidence : apres  "au moment de recrutement,
compte_bancaire_local : titulaire d'un compte bancaire, le numero de compte est de 20 chiffres.


nom_prenom_signataire : le nom de la personne qui signe en qualité de Responsable des Expartiés.
date_signature (après "Bethioua le", format jj/MM/aaaa)


        

--- Si BULLETIN ---
nom_prenom_salarie, matricule, mois_bulletin,
salaire_base, salaire_brut, retenue_ss, retenue_irg, retenue_mutuelle, net_a_payer

RÈGLES :
-   Retourne UNIQUEMENT un JSON valide, sans texte avant ni après
-   Commence toujours par {"type": "...")
-   Si un champ est absent ou illisible : null
-   Pas de markdown, pas de backticks
-   N'invente aucun champ supplémentaire
-   Lorsqu'un texte est écrit entre parenthèses dans le document, extraire exactement ce texte sans modification.
-   Ne pas reformuler, ne pas traduire, ne pas supprimer les parenthèses.
-   Si un champ demande un texte entre parenthèses, retourner uniquement le contenu exact visible.
"""

print(f'✅ Prompt : {len(PROMPT_UNIVERSEL)} caractères')

## 8. Traitement d'un PDF (batch GPU)

In [ ]:
def process_pdf(pdf_path: Path, verbose=True) -> dict:
    """
    Traite un PDF avec batch GPU.
    Toutes les pages non-blanches sont envoyées par groupes de GPU_BATCH_SIZE.
    Retourne un dict avec données + stats (tokens, temps).
    """
    import gc
    TYPES_ATTENDUS = {'OV', 'ANNEXE_II', 'BULLETIN','ANNEXE_I','AUTRE'}

    pages    = pdf_to_pages(pdf_path)
    results  = {}
    doublons = []
    total_tok_in  = 0
    total_tok_out = 0
    t_dossier     = time.time()

    if verbose:
        print(f'\n📁 {pdf_path.name} — {len(pages)} page(s)')

    # Filtrer pages non-blanches
    pages_actives = [p for p in pages if not is_blank(p['image'])]
    pages_vides   = len(pages) - len(pages_actives)

    if verbose and pages_vides:
        print(f'  {pages_vides} page(s) vide(s) ignorée(s)')

    # Traitement par batch
    for batch_start in range(0, len(pages_actives), GPU_BATCH_SIZE):
        batch  = pages_actives[batch_start:batch_start + GPU_BATCH_SIZE]
        images = [p['image'] for p in batch]

        t_batch = time.time()
        reps    = ask_batch(PROMPT_UNIVERSEL, images)
        elapsed_batch = time.time() - t_batch

        for page, rep in zip(batch, reps):
            i        = page['index']
            data     = parse_json(rep['text'])
            doc_type = data.get('type', 'INCONNU')

            total_tok_in  += rep['tokens_in']
            total_tok_out += rep['tokens_out']

            if doc_type in ('AUTRE', 'INCONNU'):
                if verbose:
                    print(f'  Page {i+1} → {doc_type} — ignorée')
                continue

            data = normalise_doc(doc_type, data)

            if verbose:
                print(f'  Page {i+1} → {doc_type:10s} '
                      f'| tok={rep["tokens_in"]}+{rep["tokens_out"]}')
                for k, v in data.items():
                    if k != 'type' and v is not None:
                        print(f'    {k:25s}: {v}')

            if doc_type in results:
                doublons.append(doc_type)
                if verbose: print(f'    ⚠️  DOUBLON {doc_type}')
                continue

            results[doc_type] = data

        # Purge VRAM entre batches
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    pages_trouvees   = sorted(results.keys())
    pages_manquantes = sorted(TYPES_ATTENDUS - set(results.keys()))

    return {
        'fichier':          pdf_path.name,
        'date_traitement':  datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'temps_total_s':    round(time.time() - t_dossier, 2),
        'tokens_in':        total_tok_in,
        'tokens_out':       total_tok_out,
        'tokens_total':     total_tok_in + total_tok_out,
        'pages_trouvees':   ', '.join(pages_trouvees),
        'pages_manquantes': ', '.join(pages_manquantes) if pages_manquantes else None,
        'anomalies':        'DOUBLON: ' + ', '.join(doublons) if doublons else None,
        **{t: results.get(t, {}) for t in TYPES_ATTENDUS},
    }


print('✅ process_pdf OK')

## 9. Export Excel

In [ ]:
SCHEMA = {
    'META': [
        ('fichier',          'Fichier'),
        ('date_traitement',  'Date traitement'),
        ('temps_total_s',    'Temps (s)'),
        ('tokens_total',     'Tokens total'),
        ('pages_trouvees',   'Pages trouvées'),
        ('pages_manquantes', 'Pages manquantes'),
        ('anomalies',        'Anomalies'),
    ],
    'OV': [
        ('monnaie',              'Monnaie'),
        ('montant_chiffres',     'Montant (chiffres)'),
        ('montant_lettres',      'Montant (lettres)'),
        ('periode',              'Période'),
        ('mois',                 'mois'),
        ('tranche',              'Tranche'),
        ('complement_ov',        'Complement'),
        ('date_demande',         'Date demande'),
        ('compte_donneur_ordre',      'compte donneur ordre'),
        ('nature_paiement_autre_libelle', 'Devise de transfert'),
        
        ('beneficiaire_nom',     'Bénéficiaire nom'),
        ('beneficiaire_compte',  'Bénéficiaire compte'),
        ('beneficiaire_adresse', 'Bénéficiaire adresse'),
        ('code_swift_banque_beneficiaire',                'SWIFT'),
        ('nom_banque_beneficiaire',           'Banque bénéficiaire'),






        
    ],
    'ANNEXE_I': [
        ('nom_prenom_employe',     'Nom Prénom client'),
        ('compte_bancaire_local',       'Compte bancaire'),
        ('nom_prenom_signataire', 'Nom Prénom signataire'),
        ('date_signature',        'Date signature'),



    ],
    'ANNEXE_II': [
        ('Periode_transfert',         'Mois transfert'),
        ('tranche_transfert',   'Tranche transfert'),
        ('complement_transfert', 'Complément transfert'),
        ('nom_prenom_travailleur',             'Nom Prénom'),
        ('compte_bancaire_local',         'Compte bancaire'),
        ('salaire_mensuel',        'Salaire mensuel'),
        ('nombre_jours',           'Nombre de jours'),
        ('nombre_jours_absence',           'nombre jours absence'),
        
        ('part_transferable',      'Part transférable'),
        ('pays_destination',       'Pays destination'),
        ('nom_banque_etranger',             'Nom banque'),
        ('numero_compte_devise_etranger', 'Compte étranger'),
        ('numero_domiciliation',   'N° Domiciliation'),



    ],
    'BULLETIN': [
        ('nom_prenom_salarie', 'Nom Prénom'),
        ('matricule',          'Matricule'),
        ('mois_bulletin',      'Mois bulletin'),
        ('salaire_base',       'Salaire base'),
        ('salaire_brut',       'Salaire brut'),
        ('retenue_ss',         'Retenue SS'),
        ('retenue_irg',        'Retenue IRG'),
        ('retenue_mutuelle',   'Retenue mutuelle'),
        ('net_a_payer',        'Net à payer'),
    ],
}

COLORS = {
    'META':      {'header': 'FF1F4E79', 'col': 'FFD6E4F0'},
    'OV':        {'header': 'FF833C00', 'col': 'FFFCE4D6'},
    'ANNEXE_I':  {'header': 'FF375623', 'col': 'FFE2EFDA'},
    'ANNEXE_II': {'header': 'FF203864', 'col': 'FFDAE3F3'},
    'BULLETIN':  {'header': 'FF3F3151', 'col': 'FFEDE7F6'},
}


 # Normalisation




         


def create_excel(path: Path, rows: list):
    wb = Workbook()
    ws = wb.active
    ws.title = 'Dossiers'
    all_cols = []
    for groupe, cols in SCHEMA.items():
        for field_key, label in cols:
            all_cols.append((groupe, field_key, label))

    col_idx = 1
    group_map = defaultdict(list)
    for groupe, _, _ in all_cols:
        group_map[groupe].append(col_idx)
        col_idx += 1

    for groupe, cols in group_map.items():
        s, e = cols[0], cols[-1]
        if s < e:
            ws.merge_cells(start_row=1, start_column=s, end_row=1, end_column=e)
        c = ws.cell(row=1, column=s)
        c.value = groupe
        c.font  = Font(bold=True, color='FFFFFFFF', name='Arial', size=11)
        c.fill  = PatternFill('solid', start_color=COLORS[groupe]['header'])
        c.alignment = Alignment(horizontal='center', vertical='center')
    ws.row_dimensions[1].height = 22

    for i, (groupe, _, label) in enumerate(all_cols, start=1):
        c = ws.cell(row=2, column=i)
        c.value = label
        c.font  = Font(bold=True, name='Arial', size=9)
        c.fill  = PatternFill('solid', start_color=COLORS[groupe]['col'])
        c.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        c.border = Border(bottom=Side(style='thin'), right=Side(style='hair'))
        ws.column_dimensions[get_column_letter(i)].width = 20
    ws.row_dimensions[2].height = 35
    ws.freeze_panes = ws.cell(row=3, column=len(SCHEMA['META']) + 1)

    for row_num, dossier in enumerate(rows, start=3):
        for col_idx, (groupe, field_key, _) in enumerate(all_cols, start=1):
            val = (dossier.get(field_key) if groupe == 'META'
                   else (dossier.get(groupe) or {}).get(field_key))
            c = ws.cell(row=row_num, column=col_idx)
            c.value = val
            c.font  = Font(name='Arial', size=9)
            c.fill  = PatternFill('solid', start_color=COLORS[groupe]['col'])
            if isinstance(val, float):
                c.number_format = '0.00'
            if groupe == 'META' and field_key in ('anomalies', 'pages_manquantes') and val:
                c.font = Font(name='Arial', size=9, bold=True, color='FFCC0000')

    wb.save(path)
    print(f'✅ Excel : {path} | {len(rows)} dossiers | {len(all_cols)} colonnes')


print('✅ Export Excel OK')

## 10. Log

In [ ]:
def log(msg: str):
    ligne = f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} — {msg}"
    print(ligne)
    with open(LOG_PATH, 'a', encoding='utf-8') as f:
        f.write(ligne + '\n')


print('✅ Log OK')

## 11. Pipeline complet

In [ ]:
import gc
import psutil

# ── Vérification RAM ──────────────────────────────────────────────────────────
ram_libre = psutil.virtual_memory().available / 1e9
print(f'RAM libre : {ram_libre:.1f} GB')

# ── Dossiers à traiter ────────────────────────────────────────────────────────
deja_traites = {f.stem for f in JSON_DIR.glob('*.json')}
a_traiter    = [p for p in pdfs if p.stem not in deja_traites]
total_pdfs   = len(pdfs)

log(f'Total PDFs     : {total_pdfs}')
log(f'Déjà traités   : {len(deja_traites)}')
log(f'À traiter      : {len(a_traiter)}')
log(f'GPU batch size : {GPU_BATCH_SIZE}')
log(f'RAM libre      : {ram_libre:.1f} GB')

# ── Chunk size automatique selon RAM disponible ───────────────────────────────
taille_img_mb = 1120 * 1584 * 3 / 1e6       # ~5MB par image
ram_estimee   = len(a_traiter) * 4 * taille_img_mb / 1e3
CHUNK_SIZE    = min(
    len(a_traiter),
    max(GPU_BATCH_SIZE, int(ram_libre * 0.7 * 1e3 / (4 * taille_img_mb)))
)
log(f'RAM estimée    : {ram_estimee:.1f} GB')
log(f'Chunk size     : {CHUNK_SIZE} dossiers')

chunks = [a_traiter[i:i+CHUNK_SIZE] for i in range(0, len(a_traiter), CHUNK_SIZE)]
log(f'{len(chunks)} chunk(s) de {CHUNK_SIZE} dossiers max')

# ── Pipeline ──────────────────────────────────────────────────────────────────
t_total       = time.time()
grand_tok_in  = 0
grand_tok_out = 0
n_ok = n_err  = 0
TYPES_ATTENDUS = {'OV', 'ANNEXE_I', 'ANNEXE_II', 'BULLETIN'}

for num_chunk, chunk in enumerate(chunks, start=1):
    log(f'── Chunk {num_chunk}/{len(chunks)} : {len(chunk)} dossiers ──')

    # ── Phase 1 : Chargement pages ────────────────────────────────────────────
    t_load    = time.time()
    all_pages = []
    for pdf_path in chunk:
        try:
            pages = pdf_to_pages(pdf_path)
            for page in pages:
                if not is_blank(page['image']):
                    all_pages.append((pdf_path, page))
        except Exception as e:
            log(f'❌ Chargement {pdf_path.name} : {e}')
    log(f'   {len(all_pages)} pages chargées en {time.time()-t_load:.1f}s')

    # ── Phase 2 : Inférence GPU ───────────────────────────────────────────────
    t_infer     = time.time()
    raw_results = defaultdict(list)

    for batch_start in range(0, len(all_pages), GPU_BATCH_SIZE):
        batch  = all_pages[batch_start:batch_start + GPU_BATCH_SIZE]
        images = [page['image'] for _, page in batch]
        try:
            reps = ask_batch(PROMPT_UNIVERSEL, images)
            for (pdf_path, page), rep in zip(batch, reps):
                raw_results[pdf_path].append({
                    'index':      page['index'],
                    'text':       rep['text'],
                    'tokens_in':  rep['tokens_in'],
                    'tokens_out': rep['tokens_out'],
                })
            pct = min(100, (batch_start + len(batch)) / len(all_pages) * 100)
            log(f'   {batch_start+len(batch):>5}/{len(all_pages)} pages | {pct:.0f}%')
        except Exception as e:
            log(f'❌ Batch {batch_start}-{batch_start+len(batch)} : {e}')
            continue
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    log(f'   Inférence en {time.time()-t_infer:.1f}s')

    # ── Phase 3 : Post-traitement + sauvegarde JSON ───────────────────────────
    t_post = time.time()

    for num, pdf_path in enumerate(chunk, start=1):
        try:
            pages_raw = raw_results.get(pdf_path, [])
            if not pages_raw:
                log(f'   [{num:>4}/{len(chunk)}] ⚠️  {pdf_path.name} — aucune page')
                continue

            results  = {}
            doublons = []
            total_tok_in  = 0
            total_tok_out = 0
            t_dossier = time.time()

            for page_raw in sorted(pages_raw, key=lambda x: x['index']):
                data     = parse_json(page_raw['text'])
                doc_type = data.get('type', 'INCONNU')
                total_tok_in  += page_raw['tokens_in']
                total_tok_out += page_raw['tokens_out']
                if doc_type in ('AUTRE', 'INCONNU'):
                    continue
                data = normalise_doc(doc_type, data)
                if doc_type in results:
                    doublons.append(doc_type)
                    continue
                results[doc_type] = data

            pages_trouvees   = sorted(results.keys())
            pages_manquantes = sorted(TYPES_ATTENDUS - set(results.keys()))

            result = {
                'fichier':          pdf_path.name,
                'date_traitement':  datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
                'temps_total_s':    round(time.time() - t_dossier, 2),
                'tokens_in':        total_tok_in,
                'tokens_out':       total_tok_out,
                'tokens_total':     total_tok_in + total_tok_out,
                'pages_trouvees':   ', '.join(pages_trouvees),
                'pages_manquantes': ', '.join(pages_manquantes) if pages_manquantes else None,
                'anomalies':        'DOUBLON: ' + ', '.join(doublons) if doublons else None,
                **{t: results.get(t, {}) for t in TYPES_ATTENDUS},
            }

            json_file = JSON_DIR / f'{pdf_path.stem}.json'
            with open(json_file, 'w', encoding='utf-8') as f:
                json.dump(result, f, ensure_ascii=False, indent=2, default=str)

            grand_tok_in  += total_tok_in
            grand_tok_out += total_tok_out
            n_ok += 1

            msg = (f'   [{num:>4}/{len(chunk)}] ✅ {pdf_path.name}'
                   f' | tok={result["tokens_total"]}'
                   f' | {result["pages_trouvees"]}')
            if result.get('pages_manquantes'):
                msg += f' | ⚠️  {result["pages_manquantes"]}'
            if result.get('anomalies'):
                msg += f' | 🔴 {result["anomalies"]}'
            log(msg)

        except Exception as e:
            n_err += 1
            log(f'   [{num:>4}/{len(chunk)}] ❌ {pdf_path.name} — {e}')
            continue

    log(f'   Post-traitement en {time.time()-t_post:.1f}s')

    # Libérer RAM du chunk
    del all_pages, raw_results
    gc.collect()
    ram_now = psutil.virtual_memory().available / 1e9
    log(f'   RAM libre après chunk : {ram_now:.1f} GB')

# ── Reconstruction Excel ──────────────────────────────────────────────────────
log('Génération Excel...')
rows = []
for json_file in sorted(JSON_DIR.glob('*.json')):
    with open(json_file, encoding='utf-8') as f:
        rows.append(json.load(f))

create_excel(EXCEL_PATH, rows)

elapsed = time.time() - t_total
log(f'✅ Terminé en {elapsed:.1f}s ({elapsed/max(1,n_ok):.1f}s/dossier)')
log(f'   Traités      : {n_ok} | Erreurs : {n_err}')
log(f'   Tokens IN    : {grand_tok_in:,}')
log(f'   Tokens OUT   : {grand_tok_out:,}')
log(f'   Tokens TOTAL : {grand_tok_in + grand_tok_out:,}')

## 12. Récupération d'urgence

Si le kernel plante et que `rows` est encore en mémoire, exécuter cette cellule
pour sauvegarder immédiatement en JSON avant de tout perdre.

In [ ]:
# Décommenter et exécuter en cas de crash kernel
# try:
#     print(f'rows en mémoire : {len(rows)}')
#     for result in rows:
#         nom = Path(result['fichier']).stem
#         jf  = JSON_DIR / f'{nom}.json'
#         if not jf.exists():
#             with open(jf, 'w', encoding='utf-8') as f:
#                 json.dump(result, f, ensure_ascii=False, indent=2, default=str)
#     print(f'✅ Sauvegardé')
# except Exception as e:
#     print(f'rows non disponible : {e}')

## 13. Analyse des résultats

In [ ]:
import pandas as pd

if EXCEL_PATH.exists():
    df = pd.read_excel(EXCEL_PATH, header=1)
    print(f'📊 {len(df)} dossiers')

    # Stats tokens
    if 'Tokens total' in df.columns:
        print(f'   Tokens total   : {df["Tokens total"].sum():,.0f}')
        print(f'   Tokens/dossier : {df["Tokens total"].mean():,.0f}')
    if 'Temps (s)' in df.columns:
        print(f'   Temps moyen    : {df["Temps (s)"].mean():.1f}s/dossier')
    print()

    if 'Anomalies' in df.columns:
        anom = df[df['Anomalies'].notna()]
        print(f'🔴 Doublons : {len(anom)}')
        if len(anom): print(anom[['Fichier','Anomalies']].to_string(index=False))
        print()

    if 'Pages manquantes' in df.columns:
        manq = df[df['Pages manquantes'].notna()]
        print(f'⚠️  Incomplets : {len(manq)}')
        if len(manq): print(manq[['Fichier','Pages manquantes']].to_string(index=False))
        print()

    display(df.head(5))
else:
    print('Excel non encore généré.')